# Deep Learning-Based Anomaly Detection using PCA and Adversarial Autoencoder

This notebook presents an anomaly detection framework for multivariate time-series data using an Adversarial Autoencoder (AAE) integrated with Principal Component Analysis (PCA).

## Supported Datasets
- SWaT
- SMAP

## Framework
- PyTorch

## Hardware
- GPU (CUDA)

# 1. Import Libraries

In [ ]:
import os
import random
import pickle
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

import sklearn
from sklearn.metrics import roc_curve,roc_auc_score, precision_recall_fscore_support

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


import torch.utils.data as data_utils

import time
from torch.cuda.amp import autocast
import copy
from torch.optim.lr_scheduler import StepLR

import gc

def get_default_device():
    """Pick GPU if available, else CPU"""
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

device = get_default_device()

In [ ]:
# Execute this cell only when running the notebook on Google Colab.
# Mount Google Drive (Google Colab only)
from google.colab import drive
drive.mount('/content/drive')

# 2. Dataset Configuration

This project supports the following datasets:

- SWaT
- SMAP

> **Note:** Datasets are not included in this repository. Please download them from their official sources and update the dataset path.

In [ ]:
def set_seed(seed=2):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)               # for current GPU
    torch.cuda.manual_seed_all(seed)           # for multi-GPU
    torch.backends.cudnn.deterministic = True  # slower but deterministic
    torch.backends.cudnn.benchmark = False     # disables optimization for reproducibility


In [ ]:
def load_data(path, name):

  with open(os.path.join(path, name), "rb") as file:
    pkl = pickle.load(file)
    print(name + " data loaded from drive ")
  return pkl

In [ ]:
def load_dataset(dataset):
  # Supported datasets:
  # SWaT
  # SMAP
  # Dataset is not included in this repository.
  # Please update the path to your local dataset.
  datasets = ['SWAT/', 'SMAP/']
  DATASET_ROOT = "/path/to/datasets"
  datasets_folder = os.path.join(DATASET_ROOT, dataset)
  if(dataset == 'SWAT'):
    train = load_data(datasets_folder, 'SWAT_train_normalized.pkl')
    test = load_data(datasets_folder, 'SWAT_test_normalized.pkl')
    labels = load_data(datasets_folder, 'SWAT_labels_normalized.pkl')
    train_id = None
    test_id = None
    train = np.array(train)
    test = np.array(test)
    labels = np.array(labels)

  elif(dataset == 'SMAP'):
    train = load_data(datasets_folder, 'SMAP_train.pkl')
    test = load_data(datasets_folder, 'SMAP_test.pkl')
    train_id = load_data(datasets_folder, 'SMAP_train_id.pkl')
    test_id = load_data(datasets_folder, 'SMAP_test_id.pkl')
    labels = load_data(datasets_folder, 'SMAP_test_label.pkl')

  else:
    raise ValueError("Invalid dataset name")

  return train, test, labels, train_id, test_id




In [ ]:
def get_dataset_parameters(dataset):
  if(dataset == 'SWAT'):
    Batch_size = 250
    sequence_length = 10
    features = 51
    latent_size = 250
    learning_rate = 1e-5
    req_prepr = False

  elif(dataset == 'SMAP'):
    Batch_size = 512
    sequence_length = 5
    features = 25
    latent_size = 275
    learning_rate = 1e-4
    req_prepr = True

  else:
    raise ValueError("Invalid dataset name")
  return Batch_size, sequence_length, features, latent_size, learning_rate, req_prepr

def normalize(a):
	min_a, max_a = np.min(a), np.max(a)
	return (a - min_a) / (max_a - min_a + 0.0001)

def preprocess(df):
    """returns normalized and standardized data.
    """

    df = np.asarray(df, dtype=np.float32)

    if len(df.shape) == 1:
        raise ValueError('Data must be a 2-D array')

    if np.any(sum(np.isnan(df)) != 0):
        print('Data contains null values. Will be replaced with 0')
        df = np.nan_to_num()

    df = normalize(df)
    #print('Data normalized')

    return df


def sliding_window(data, window_size):

  n_samples, n_features = data.shape
  windows = [data[i:i+window_size] for i in range(n_samples - window_size + 1)]
  return np.array(windows)

#dataloader
def create_train_dataloader(x_train, Batch_size):

  windows_normal_train = x_train[:int(np.floor(.8 *  x_train.shape[0]))]
  windows_normal_val = x_train[int(np.floor(.8 *  x_train.shape[0])):int(np.floor(x_train.shape[0]))]
  #convert datasets from array to tensor dataset
  train_dataset = data_utils.TensorDataset(torch.tensor(windows_normal_train, dtype=torch.float32))
  val_dataset =  data_utils.TensorDataset(torch.tensor(windows_normal_val, dtype=torch.float32))
  train_loader =data_utils.DataLoader(train_dataset , batch_size=Batch_size, shuffle=False )
  val_loader = data_utils.DataLoader(val_dataset , batch_size=Batch_size, shuffle=False)
  return train_loader, val_loader

def create_test_dataloader(x_test, Batch_size):

  test_dataset= data_utils.TensorDataset(torch.tensor(x_test, dtype=torch.float32))
  test_loader = data_utils.DataLoader(test_dataset , batch_size=Batch_size, shuffle=False)
  return test_loader


def plot_losses( validate):

  plt.plot(validate, '-x', label="validate_loss")
  plt.xlabel('epoch')
  plt.ylabel('loss')
  plt.legend()
  plt.title('Losses vs. No. of epochs')
  plt.grid()
  plt.show()

# 3. Training Configuration

In [ ]:
#parameters
#dataset: SWAT, SMAP
dataset= 'SMAP'
train, test, labels, train_id, test_id = load_dataset(dataset)
Batch_size, sequence_length, features, latent_size, learning_rate, req_prepr = get_dataset_parameters(dataset)
epochs = 100
loss_fun_train = nn.MSELoss()
loss_fun_test = nn.MSELoss(reduction='none')


In [ ]:
if(req_prepr):#if the data not normalized
  train = preprocess(train)
  test = preprocess(test)

train_w = sliding_window(train, sequence_length)
test_w = sliding_window(test, sequence_length)

train_loader, val_loader = create_train_dataloader(train_w, Batch_size)

# 4. Model Architecture

In [ ]:
'''
for SWAT dataset: seq_1*51, 375, 375, latent(250)
for SMAP dataset: seq_1*25, 64, 32, latent(275)
'''


'''
#Autoencoder model SWaT
class Encoder(nn.Module):
  def __init__(self, seq_l, latent_size):
    super().__init__()
    self.encoder = nn.Sequential(
      nn.Dropout(0.2),
      nn.Linear(seq_l *51, 255 ),
      nn.BatchNorm1d(255),
      nn.ReLU(),
      nn.Linear(255,125),
      nn.BatchNorm1d(125),
      nn.ReLU(),
      nn.Linear(125, latent_size)
      )
  def forward(self, x):
    batch_size = x.size(0)
    x = x.view(batch_size, -1)
    x = self.encoder(x)
    return x



class Decoder(nn.Module):
  def __init__(self, seq_l, latent_size):
    super().__init__()
    self.seq_length = seq_l
    self.decoder = nn.Sequential(
      nn.Linear(latent_size,125),
      nn.ReLU(),
      nn.Linear(125, 255),
      nn.ReLU(),
      nn.Linear(255,seq_l* 51),
      nn.Sigmoid()
        )
  def forward(self, x):
    x = self.decoder(x)
    batch_size = x.size(0)
    x = x.view(batch_size, self.seq_length, 51)
    return x '''


#Autoencoder model SMAP
class Encoder(nn.Module):
  def __init__(self, seq_l, latent_size):
    super().__init__()
    self.encoder = nn.Sequential(
      nn.Dropout(0.2),
      nn.Linear(seq_l *25, 64 ),
      nn.BatchNorm1d(64),
      nn.ReLU(),
      nn.Linear(64,32),
      nn.BatchNorm1d(32),
      nn.ReLU(),
      nn.Linear(32, latent_size)
      )
  def forward(self, x):
    batch_size = x.size(0)
    x = x.view(batch_size, -1)
    x = self.encoder(x)
    return x



class Decoder(nn.Module):
  def __init__(self, seq_l, latent_size):
    super().__init__()
    self.seq_length = seq_l
    self.decoder = nn.Sequential(
      nn.Linear(latent_size,32),
      nn.ReLU(),
      nn.Linear(32, 64),
      nn.ReLU(),
      nn.Linear(64,seq_l* 25),
      nn.Sigmoid()
        )
  def forward(self, x):
    x = self.decoder(x)
    batch_size = x.size(0)
    x = x.view(batch_size, self.seq_length, 25)
    return x




class AE(nn.Module):
  def __init__(self, seq_len , latent_size):
    super().__init__()
    #self.s = seq_len
    self.encoder = Encoder(seq_len, latent_size)
    self.decoder = Decoder(seq_len, latent_size)

class Discriminator(nn.Module):
    def __init__(self, latent_size):
        super(Discriminator, self).__init__()
        self.fc1 = nn.Linear( latent_size, int(latent_size/2))
        self.fc2 = nn.Linear( int(latent_size/2), int(latent_size/2))
        self.fc3 = nn.Linear(int(latent_size/2), 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

class AAE(nn.Module):
  def __init__(self, seq_l, latent_size):
    super().__init__()
    self.encoder = Encoder(seq_l, latent_size)
    self.decoder = Decoder(seq_l, latent_size)
    self.discriminator = Discriminator(latent_size)


# 5. Principal Component Analysis (PCA)

In [ ]:
def pad_batch(batch, target_size):
    current_size = batch.size(0)
    if current_size < target_size:
        padding_size = target_size - current_size
        padding = torch.zeros(padding_size, batch.size(1),  device=batch.device)
        padded_batch = torch.cat([batch, padding], dim=0)
        return padded_batch
    return batch

def apply_PCA(latent):
  pca = PCA()

  #check last batch size
  if(latent.shape[0]<latent.shape[1]):
    original_size = latent.shape[0]
    latent = pad_batch(latent, latent.shape[1])
    latent_np = latent.detach().cpu().numpy()
    latent_pca = pca.fit_transform(latent_np)
    #latent_denoised = pca.inverse_transform(latent_pca)

    #convert back to a tensor
    latent_pca_tensor = torch.tensor(latent_pca, dtype=torch.float32)
    latent_pca_tensor = latent_pca_tensor[:original_size, :]

  else:
    latent_np = latent.detach().cpu().numpy()
    latent_pca = pca.fit_transform(latent_np)
    #latent_denoised = pca.inverse_transform(latent_pca)

    #convert back to a tensor
    latent_pca_tensor = torch.tensor(latent_pca, dtype=torch.float32)

  # Ensure tensor is on the same device as the original latent tensor
  latent_pca_tensor = latent_pca_tensor.to(latent.device)
  return latent_pca_tensor

#6. Training Functions

In [ ]:
def train(model, train_loader, optimizer,loss_fun):
  model.train()

  for batch_idx, data in enumerate(train_loader):
    batch = data[0].to(device)
    optimizer.zero_grad()

    latent = model.encoder(batch)
    #latent = apply_PCA(latent)
    outputs = model.decoder(latent)

    loss = loss_fun(outputs, batch)
    loss.backward()
    optimizer.step()

def evaluate(model, val_loader, loss_fun):
  model.eval()
  eval_loss = []
  with torch.no_grad():
    for batch_idx, data in enumerate(val_loader):
      batch = data[0].to(device)

      latent = model.encoder(batch)
      #latent = apply_PCA(latent)
      outputs = model.decoder(latent)

      loss = loss_fun(outputs, batch)
      eval_loss.append(loss.cpu().numpy())

  return np.mean(eval_loss)




def AE_training(model, train_loader, eval_loader, num_epochs, learning_rate, loss_fun):
    loss_list = []
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):
      train(model,train_loader,optimizer, loss_fun)
      eva_result= evaluate(model, eval_loader, loss_fun)
      print(f'Epoch {epoch}, Loss: {eva_result}')
      loss_list.append(eva_result)
    return loss_list

def training_scores(model, validate_loader,loss_fun, alpha =1, beta =0.2):

  results=[]
  model.eval()
  with torch.no_grad():
    pred_time = []

    for batch_idx, data in enumerate(validate_loader):
      start_time = time.time()
      batch_data = data[0].to(device)
      latent = model.encoder(batch_data)
      #latent = apply_PCA(latent)
      output = model.decoder(latent)
      batch_loss = loss_fun(output, batch_data)
      loss_per_sequence = batch_loss.mean(dim=[1, 2])  # Resulting shape: [batch_size]

      #discr = model.discriminator(latent)
      #adv_loss = (1 - discr).squeeze()  # Shape: [batch_size]
      #total_loss = alpha * loss_per_sequence + beta * adv_loss

      results.extend(loss_per_sequence.tolist())
      pred_time.append(time.time() - start_time)

    #results = (results - np.min(results)) / (np.max(results) - np.min(results))
  return results, np.mean(pred_time)



In [ ]:
def plot_pca_variance(latent, epoch):
    # Convert to numpy
    latent_np = latent.detach().cpu().numpy()

    # Apply PCA
    pca = PCA()
    pca.fit(latent_np)

    explained = pca.explained_variance_ratio_
    cumulative = explained.cumsum()

    threshold = 0.99
    n_components = np.argmax(cumulative >= threshold) + 1

    # Plot
    plt.figure(figsize=(6, 4))
    #plt.plot(range(1, len(explained) + 1), explained, marker='o', linestyle='--', label='Explained Variance')
    plt.plot(range(1, len(cumulative) + 1), cumulative, marker='s', linestyle='--', label='Cumulative Variance')
    plt.axhline(y=1, color='red', linestyle=':', label=f'99% of variance in {n_components} components ' )
    plt.title(f'PCA Explained Variance (Epoch {epoch+1})')
    plt.xlabel('Number of Components')
    plt.ylabel('Variance Ratio')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def AAE_training(model, train_loader, eval_loader, num_epochs, learning_rate, withPCA = False, save_points = None):

  #learning_rate = 0.00001

  # Optimizers
  ae_optimizer = torch.optim.Adam(list(model.encoder.parameters()) + list(model.decoder.parameters()),  lr=learning_rate, betas=(0.9, 0.999))
  g_optimizer = torch.optim.Adam(model.encoder.parameters() , lr=learning_rate, betas=(0.1, 0.999))
  d_optimizer = torch.optim.Adam(model.discriminator.parameters(), lr=learning_rate, betas=(0.1, 0.999))

  '''ae_scheduler = StepLR(ae_optimizer, step_size= 20, gamma=0.1)
  g_scheduler = StepLR(g_optimizer, step_size= 20, gamma=0.1)
  d_scheduler = StepLR(d_optimizer, step_size= 20, gamma=0.1)'''

  # Loss functions
  reconstruction_loss = nn.MSELoss()
  adversarial_loss = nn.BCELoss()

  lambda_adv = 0.5

  train_loss_list = []
  eval_loss_list = []

# Training process
  start_time = time.time()
  train_time = []

  lr_models = []
  '''best_loss = float('inf')
  patience = 15
  counter = 0'''

  plot_points = [0, 19, 49, 69, 99]

  for epoch in range(num_epochs):
    epoch_start_time = time.time()
    model.encoder.train()
    model.decoder.train()
    model.discriminator.train()
    train_loss_batches = []
    eval_loss_batches = []

    for batch_idx, data in enumerate(train_loader):
      ae_optimizer.zero_grad()
      batch = data[0].to(device)
      fake_samples = model.encoder(batch).to(device)
      #fake_samples = apply_PCA(fake_samples) # PCA over All the model
      fake_samples_adv = fake_samples.detach().to(device)
      fake_samples_adv = apply_PCA(fake_samples_adv)
      fake_samples_gen = fake_samples_adv.detach().to(device)
      #fake_samples_gen = apply_PCA(fake_samples_gen)
      latent_size = fake_samples_gen.size(1)

      #print components in plot
      if epoch in plot_points:
        if batch_idx == 1:
          plot_pca_variance(fake_samples, epoch)

      #autoencoder trainig
      #if(withPCA):
      #fake_samples = apply_PCA(fake_samples)# PCA over first phase

      reconstructed_batch = model.decoder(fake_samples)
      rec_loss = reconstruction_loss(reconstructed_batch, batch)


      # Compute adversarial loss
      real_labels = torch.ones(batch.size(0), 1).to(device)
      fake_labels = torch.zeros(batch.size(0), 1).to(device)

      validity_fake_z_adv = model.discriminator(fake_samples_adv).to(device)
      adv_loss = adversarial_loss(validity_fake_z_adv, real_labels).to(device)

      # Combine losses using lambda
      total_ae_loss = rec_loss + lambda_adv * adv_loss
      total_ae_loss.backward(create_graph=False, retain_graph=False)
      ae_optimizer.step()

      # Adversarial training
      model.encoder.eval()
      d_optimizer.zero_grad()

      real_samples = torch.normal(mean=0, std=1, size=(batch.size(0), fake_samples.size(1))).to(device) # Gaussian distribution
      real_samples = apply_PCA(real_samples)
      real_loss = adversarial_loss(model.discriminator(real_samples), real_labels).to(device)
      fake_loss = adversarial_loss(model.discriminator(fake_samples_adv), fake_labels).to(device)
      d_loss = 0.5 * (real_loss + fake_loss).to(device)
      d_loss.backward(create_graph=False, retain_graph=False)
      d_optimizer.step()

      # Generator training
      model.encoder.train()
      g_optimizer.zero_grad()
      validity_fake_z_gen = model.discriminator(fake_samples_gen) ###########
      g_loss = adversarial_loss(validity_fake_z_gen, real_labels).to(device)
      g_loss.backward(create_graph=False, retain_graph=False)
      g_optimizer.step()
      train_loss_batches.append(rec_loss.detach().cpu().numpy())





    train_loss_list.append(np.mean(train_loss_batches))
    train_time.append(time.time() - epoch_start_time)

    # Evaluation step
    model.encoder.eval()
    model.decoder.eval()
    model.discriminator.eval()
    eval_loss_batches = []
    with torch.no_grad():
      for batch_idx, data in enumerate(eval_loader):
        batch = data[0].to(device)

        fake_samples = model.encoder(batch)
        fake_samples = fake_samples.to(device)

        #AE loss
        decoded_x = model.decoder(fake_samples)  #########
        AE_loss = reconstruction_loss(decoded_x, batch).to(device)
        eval_loss_batches.append(AE_loss.detach().cpu().numpy())
    eval_loss_list.append(np.mean(eval_loss_batches))
    #print(f"Epoch [{epoch+1}/{num_epochs}],  validate Loss: {np.mean(eval_loss_batches)}")

    '''ae_scheduler.step()
    g_scheduler.step()
    d_scheduler.step()'''

    if epoch in save_points:
        # Save a deep copy of the model
        model_copy = copy.deepcopy(model)
        lr_models.append(model_copy)

    '''#early stopping
    if eval_loss_list[-1] < best_loss:
        best_loss = eval_loss_list[-1]
        counter = 0  # Reset patience
    else:
        counter += 1
        if counter >= patience:
            print(f"Stopping early at epoch {epoch+1}")
            break'''

  total_train_time = time.time() - start_time
  one_epoch_time = np.mean(train_time)
  return total_train_time, one_epoch_time , eval_loss_list, lr_models

# 7. Model Training

In [ ]:
# Train the model on the selected dataset
save_points = [50, 75, 90]
model = AAE(sequence_length, latent_size).to(device)
train_time, epoch_time, validate_loss, lr_models = AAE_training(model, train_loader, val_loader, num_epochs=100, learning_rate = 1e-4, save_points= save_points)

In [ ]:
plot_losses(validate_loss)

In [ ]:
#select best model
model = lr_models[11]

#save the trained model
torch.save(model.state_dict(), 'saveFolderPath/modelName.pt')


# 8. Model Evaluation

In [ ]:
#eval_methods

def calc_point2point(predict, actual):

    TP = np.sum(predict * actual)
    TN = np.sum((1 - predict) * (1 - actual))
    FP = np.sum(predict * (1 - actual))
    FN = np.sum((1 - predict) * actual)
    precision = TP / (TP + FP + 0.00001)
    recall = TP / (TP + FN + 0.00001)
    f1 = 2 * (precision * recall) / (precision + recall + 0.00001)
    return f1, precision, recall, TP, TN, FP, FN


def adjust_predicts(score, label, adjustment,
                    threshold=None,
                    pred=None,
                    calc_latency=False):

    if len(score) != len(label):
        raise ValueError("score and label must have the same length")
    score = np.asarray(score)
    label = np.asarray(label)
    latency = 0

    if pred is None:
        predict = score > threshold
    else:
        predict = pred

    if(adjustment):
      actual = label > 0.1
      anomaly_state = False
      anomaly_count = 0
      for i in range(len(score)):
          if actual[i] and predict[i] and not anomaly_state:
                  anomaly_state = True
                  anomaly_count += 1
                  for j in range(i, 0, -1):
                      if not actual[j]:
                          break
                      else:
                          if not predict[j]:
                              predict[j] = True
                              latency += 1
          elif not actual[i]:
              anomaly_state = False
          if anomaly_state:
              predict[i] = True
      if calc_latency:
          return predict, latency / (anomaly_count + 1e-4)
      else:
          return predict

    else:

      if calc_latency:
          # If latency calculation is requested, return 0 since point adjustment is disabled
          return predict, 0
      else:
          return predict

def calc_seq(score, label, threshold, adjustment, calc_latency=True):

    if calc_latency:
        predict, latency = adjust_predicts(score, label, adjustment, threshold, calc_latency=calc_latency)
        t = list(calc_point2point(predict, label))
        t.append(latency)
        return t, predict
    else:
        predict = adjust_predicts(score, label, adjustment, threshold, calc_latency=calc_latency)
        return calc_point2point(predict, label), predict


def bf_search(score, label, adjustment= True):

    predict = 0.0
    m = (-1., -1., -1.)
    m_t = 0.0
    for i in range(98):
        threshold  = 0.01 * i + 0.01
        target, predict1 = calc_seq(score, label, threshold, adjustment, calc_latency=True)
        if target[0] > m[0]:
            m_t = threshold
            m = target
            predict = predict1


    return m, m_t, predict #,point_label

def prediction_result(model, test_loader, test_label, loss_fun_test ):
  y_pred, pred_time = training_scores(model, test_loader, loss_fun_test)
  y_test = test_label[-len(y_pred):]

  #result
  #with point adjustment
  t, threshold, predicted = bf_search(y_pred, y_test, adjustment= True)

  '''print("with point adjustment:" )
  print('best_f1:', t[0], 'pre:', t[1], 'rec:', t[2], 'TP:', t[3], 'TN:', t[4], 'FP:', t[5], 'FN:', t[6],
              'latency:', t[7], 'threshold:', threshold)

  #without point adjustment
  t2, threshold2, predicted2 = bf_search(y_pred, y_test, adjustment= False)
  print("without point adjustment:" )
  print('best_f1:', t2[0], 'pre:', t2[1], 'rec:', t2[2], 'TP:', t2[3], 'TN:', t2[4], 'FP:', t2[5], 'FN:', t2[6],
              'latency:', t2[7], 'threshold:', threshold2)'''
  return t, threshold, predicted

In [ ]:
# Evaluate the trained model on the test dataset
test_loader = create_test_dataloader(test_w, Batch_size)
t, threshold, predicted_labels = prediction_result(model, test_loader, labels, loss_fun_test)


print('best_f1:', t[0], 'pre:', t[1], 'rec:', t[2], 'TP:', t[3], 'TN:', t[4], 'FP:', t[5], 'FN:', t[6],
              'latency:', t[7], 'threshold:', threshold)


In [ ]:
#for dataset with multiple entites, SMAP, train and test each entity separately
entity_ids = np.intersect1d(np.unique(train_id), np.unique(test_id))
results = []



model = AAE(sequence_length, latent_size).to(device)

for eid in np.unique(test_id):
    x_train = train[train_id == eid]
    x_test = test[test_id == eid]
    y_test = labels[test_id == eid]

    x_train_w = sliding_window(x_train, sequence_length)
    x_test_w = sliding_window(x_test, sequence_length)

    train_loader, val_loader = create_train_dataloader(x_train_w, Batch_size)
    test_loader = create_test_dataloader(x_test_w, Batch_size)

    #train the model
    train_time, epoch_time, validate_loss, lr_models = AAE_training(model, train_loader, val_loader, num_epochs=100, learning_rate = 1e-4, save_points= save_points)
    train_time, epoch_time, validate_loss, _ = AAE_training(model, train_loader, val_loader, epochs, learning_rate )

    # Evaluate the trained model
    t, threshold, predicted_labels = prediction_result(model, test_loader, y_test, loss_fun_test)

    results.append({
        'entity_id': eid,
        'f1': t[0],
        'precision': t[1],
        'recall': t[2],
        'threshold': threshold,
        'TP': t[3],
        'TN': t[4],
        'FP': t[5],
        'FN': t[6],
        'latency': t[-1],
        'train_time': train_time
    })
    print("entity id ", eid, ", F1: ", t[0])





In [ ]:
# Print summary
for r in results:
    print(f"Entity {r['entity_id']} | F1: {r['f1']:.4f} | P: {r['precision']:.4f} | R: {r['recall']:.4f} | Th: {r['threshold']:.3f}")

# Assuming each res dict contains 'tp', 'fp', 'fn' keys (counts per context)
# If your results don't contain these, you need to compute them from predictions and labels.

# Sum TP, FP, FN over all contexts
total_tp = sum(res['TP'] for res in results)
total_fp = sum(res['FP'] for res in results)
total_fn = sum(res['FN'] for res in results)

# Calculate micro precision, recall, F1
micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
micro_f1 = (
    2 * micro_precision * micro_recall / (micro_precision + micro_recall)
    if (micro_precision + micro_recall) > 0
    else 0
)

# Print micro-averaged results
print("Micro-Averaged F1 Score:", round(micro_f1, 4))
print("Micro-Averaged Precision:", round(micro_precision, 4))
print("Micro-Averaged Recall:", round(micro_recall, 4))

In [ ]:
# Extract each metric
f1_scores = [res['f1'] for res in results]
precisions = [res['precision'] for res in results]
recalls = [res['recall'] for res in results]

# Compute averages
avg_f1 = np.mean(f1_scores)
avg_precision = np.mean(precisions)
avg_recall = np.mean(recalls)

# Print results
print("Average F1 Score:", round(avg_f1, 4))
print("Average Precision:", round(avg_precision, 4))
print("Average Recall:", round(avg_recall, 4))
